## Importing Libraries

In [ ]:

# Import python packages
import streamlit as st
import pandas as pd
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, lit, when, regexp_replace
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, TimestampType
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
import time
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, lit, when, regexp_replace
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, TimestampType
import pandas as pd
import requests
import json
from datetime import datetime, timedelta
import time
from textwrap import dedent

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
SELECT CURRENT_REGION() AS acct_region;
-- SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT;


## Quick checks & fixes

In [ ]:
SELECT ENTITY_NAME, COUNT(*) FROM SIGNAL_EXTRACTION_DB.RAW.NEWS_ARTICLES_DATA GROUP BY ENTITY_NAME;

In [ ]:
SELECT * FROM SIGNAL_EXTRACTION_DB.RAW.NEWS_ARTICLES_DATA WHERE ENTITY_NAME = 'Palantir' LIMIT 3;

## Main Code

In [ ]:
def create_signal_features_with_task_description(session, source_table="SIGNAL_EXTRACTION_DB.RAW.NEWS_ARTICLES_DATA", target_table="SIGNAL_EXTRACTION_DB.STAGING.NEWS_ARTICLES_FEATURES"):
    """
    Build news signal features using custom LLM Task Descriptions

    Includes:
      - SENTIMENT_IMPACT_SCORE (1..5)
      - EVENT_TYPE (string)
      - NEWS_RELIABILITY (string)
      - RELEVANCE_CLASS (string)
      - NOVELTY_CLASS (string)
      - SECTOR (string)
    """
    feature_sql = dedent(f"""
        CREATE OR REPLACE TABLE {target_table} AS
        WITH
        
        RAW_EXTRACT AS (
            SELECT * FROM {source_table}
        ),
        
        /* 1) Prepare base article rows and text to score */
        BASE AS (
            SELECT
                ARTICLE_ID,
                SOURCE_NAME,
                TITLE,
                SUBSTR(
                    COALESCE(TITLE,'') || ' ' ||
                    COALESCE(DESCRIPTION,'') || ' ' ||
                    COALESCE(CONTENT_FULL,''),
                    1, 40000
                ) AS FULL_TEXT,
                ENTITY_NAME::STRING
            FROM {source_table} 
        ),

        /* 2) Sentiment Impact (1–5 via mapping) */
        SENTIMENT AS (
            SELECT
                ARTICLE_ID,
                AI_CLASSIFY(
                    FULL_TEXT,
                    ARRAY_CONSTRUCT('very_bearish_for_stock','bearish_for_stock','neutral','bullish_for_stock','very_bullish_for_stock'),
                    OBJECT_CONSTRUCT('task_description',
                        'Classify the article’s impact on the specified stock price for trading. Pick exactly one sentiment label for the entity_name or stock.'
                    )
                ) AS SENTIMENT_OBJ
            FROM BASE
        ),

        /* 3) Event Type */
        EVENT AS (
            SELECT
                ARTICLE_ID,
                AI_CLASSIFY(
                    FULL_TEXT,
                    ARRAY_CONSTRUCT(
                        'earnings_results','mergers_and_acquisitions','regulatory_litigation',
                        'new_product_release','operational_disruption','geopolitics',
                        'management_change','analyst_rating','other'
                    ),
                    OBJECT_CONSTRUCT('task_description',
                        'Identify the main event type most likely to drive price action for the stock. Choose a single label.'
                    )
                ) AS EVENT_OBJ
            FROM BASE
        ),

        /* 4) Reliability */
        RELIAB AS (
            SELECT
                ARTICLE_ID,
                AI_CLASSIFY(
                    FULL_TEXT,
                    ARRAY_CONSTRUCT('factual','speculation','mixed'),
                    OBJECT_CONSTRUCT('task_description',
                        'Judge the reliability of the article’s claims. Select whether information is factual, speculative, or mixed.'
                    )
                ) AS RELIAB_OBJ
            FROM BASE
        ),

        /* 5) Relevance to RMBS (scope) */
        RELEVANCE AS (
            SELECT
                ARTICLE_ID,
                AI_CLASSIFY(
                    FULL_TEXT,
                    ARRAY_CONSTRUCT('about_entity','about_sector_peer','sector_wide','market_wide','unrelated'),
                    OBJECT_CONSTRUCT('task_description',
                        'Decide how directly the article relates to the specified stock. Select the single scope label.'
                    )
                ) AS RELEVANCE_OBJ
            FROM BASE
        ),

        /* 6) Novelty of information */
        NOVELTY AS (
            SELECT
                ARTICLE_ID,
                AI_CLASSIFY(
                    FULL_TEXT,
                    ARRAY_CONSTRUCT('novel_material','incremental_update','duplicate_repackaged','opinion_commentary'),
                    OBJECT_CONSTRUCT('task_description',
                        'Assess whether the article provides new material, an incremental update, a duplicate, or opinion content.'
                    )
                ) AS NOVELTY_OBJ
            FROM BASE
        ),

        /* 7) Sector classification (GICS-like) */
        SECTOR AS (
            SELECT
                ARTICLE_ID,
                AI_CLASSIFY(
                    FULL_TEXT,
                    ARRAY_CONSTRUCT(
                        'information_technology','communication_services','consumer_discretionary','semiconductor',
                        'consumer_staples','health_care','financials','industrials',
                        'energy','materials','utilities','real_estate','multi_sector_or_other'
                    ),
                    OBJECT_CONSTRUCT('task_description',
                        'Assign the stock’s sector based on the article. Choose exactly one sector label.'
                    )
                ) AS SECTOR_OBJ
            FROM BASE
        )

        /* 8) Final projection */
        SELECT
            rw.ENTITY_NAME,
            rw.PUBLISHED_AT_UTC,
            CAST(rw.PUBLISHED_AT_UTC AS DATE) AS PUBLISHED_DATE,
            b.ARTICLE_ID,
            b.TITLE,
            rw.DESCRIPTION,
            rw.CONTENT_SIZE,
            rw.CONTENT_FULL,
            b.FULL_TEXT,
            rw.AUTHOR,
            b.SOURCE_NAME,
            rw.SOURCE_ID,
            rw.URL,
            rw.URL_TO_IMAGE,
            rw.URL_DOMAIN,
            rw.CONTENT_TRUNCATED,
            rw.INGESTED_AT AS INGESTED_AT_NEWS_ARTICLES_DATA,

            /* Sentiment score mapping from top label */
            CASE s.SENTIMENT_OBJ:labels[0]::STRING
                WHEN 'very_bearish_for_stock' THEN 1
                WHEN 'bearish_for_stock'      THEN 2
                WHEN 'neutral'                THEN 3
                WHEN 'bullish_for_stock'      THEN 4
                WHEN 'very_bullish_for_stock' THEN 5
                ELSE NULL
            END                                                             AS SENTIMENT_IMPACT_SCORE,

            e.EVENT_OBJ:labels[0]::STRING                                   AS EVENT_TYPE,
            r.RELIAB_OBJ:labels[0]::STRING                                  AS NEWS_RELIABILITY,
            v.RELEVANCE_OBJ:labels[0]::STRING                               AS RELEVANCE_CLASS,
            n.NOVELTY_OBJ:labels[0]::STRING                                 AS NOVELTY_CLASS,
            sct.SECTOR_OBJ:labels[0]::STRING                                AS SECTOR,

            CURRENT_TIMESTAMP()                                             AS INGESTED_AT_NEWS_ARTICLES_FEATURES
        FROM BASE b
        LEFT JOIN RAW_EXTRACT rw ON b.ARTICLE_ID = rw.ARTICLE_ID
        LEFT JOIN SENTIMENT s  ON b.ARTICLE_ID = s.ARTICLE_ID
        LEFT JOIN EVENT e      ON b.ARTICLE_ID = e.ARTICLE_ID
        LEFT JOIN RELIAB r     ON b.ARTICLE_ID = r.ARTICLE_ID
        LEFT JOIN RELEVANCE v  ON b.ARTICLE_ID = v.ARTICLE_ID
        LEFT JOIN NOVELTY n    ON b.ARTICLE_ID = n.ARTICLE_ID
        LEFT JOIN SECTOR sct   ON b.ARTICLE_ID = sct.ARTICLE_ID
    """)

    session.sql(feature_sql).collect()
    return session.table(target_table)

print("\n Generating signals using Snowflake Cortex...")
feature_df = create_signal_features_with_task_description(session)
print("Complete")

In [ ]:
feature_df.limit(3).show()

In [ ]:
SELECT * FROM SIGNAL_EXTRACTION_DB.STAGING.NEWS_ARTICLES_FEATURES LIMIT 3;

In [ ]:
# compelte